# RC1 — Data Quality & Bar Analysis

**Research Checkpoint 1** for the RSPCP Bachelor's Thesis.

This notebook validates data quality after Phases 1–2 (OHLCV ingestion + Lopez de Prado alternative bars) and makes go/no-go decisions before feature engineering (Phase 4).

**Sections:**
1. Data Coverage — heatmap, gap detection, asset filtering
2. Bar Construction — bar counts, duration statistics
3. Return Distributions — descriptive stats, QQ plots, normality tests
4. Autocorrelation — ACF/PACF, Ljung-Box, volatility clustering
5. Conclusions — summary table and go/no-go decisions

In [ ]:
# Standard library
from __future__ import annotations

import os
import sys
import warnings
from pathlib import Path

# Ensure we run from the project root so that:
#   1) `src.*` imports resolve correctly
#   2) .env is found by pydantic-settings (DUCKDB_PATH may be relative)
_PROJECT_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd().parent
os.chdir(_PROJECT_ROOT)
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Third-party
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
from bokeh.io import output_notebook, show
from bokeh.layouts import column

output_notebook()
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:.6f}".format)
print(f"Working directory: {Path.cwd()}")

In [ ]:
# Project imports — analysis services
from src.app.research.application.autocorrelation_analyzer import AutocorrelationAnalyzer
from src.app.research.application.bar_comparator import BarComparator
from src.app.research.application.coverage_analyzer import CoverageAnalyzer
from src.app.research.application.data_loader import DataLoader
from src.app.research.application.return_analyzer import ReturnAnalyzer

# Project imports — chart functions
from src.app.research.infrastructure.bokeh_charts import (
    create_bar_count_histogram,
    create_coverage_heatmap,
    create_gap_timeline,
    create_statistics_table,
    create_volume_profile,
)
from src.app.research.infrastructure.matplotlib_charts import (
    create_acf_comparison_grid,
    create_acf_stem_plot,
    create_bar_duration_boxplot,
    create_qq_grid,
    create_return_distribution,
)

# Project imports — infrastructure
from src.app.system.database.connection import ConnectionManager

In [ ]:
# Initialise database connection and all analysis services
cm = ConnectionManager()
cm.initialize()

loader = DataLoader(cm)
coverage_analyzer = CoverageAnalyzer(loader)
return_analyzer = ReturnAnalyzer()
acf_analyzer = AutocorrelationAnalyzer()
bar_comparator = BarComparator()

print("Services initialised successfully.")

## 1. Data Coverage

Check which assets and timeframes are available, compute coverage percentages, detect gaps, and filter assets that meet minimum quality requirements.

In [ ]:
# Discover available assets and timeframes
assets = loader.get_available_assets()
timeframes = ["1h", "4h", "1d"]

print(f"Available assets ({len(assets)}): {assets}")
print(f"Timeframes to analyse: {timeframes}")

In [ ]:
# Compute coverage for all (asset, timeframe) pairs
coverage_records = coverage_analyzer.compute_coverage(assets, timeframes)
coverage_df = pd.DataFrame([r.model_dump() for r in coverage_records])
coverage_df

In [ ]:
# Coverage heatmap (interactive Bokeh)
coverage_matrix = coverage_analyzer.build_coverage_matrix(coverage_records)
heatmap = create_coverage_heatmap(coverage_matrix)
show(heatmap)

In [ ]:
# Gap detection — find significant data gaps for each asset (1h timeframe)
all_gaps = []
for asset in assets:
    gaps = coverage_analyzer.detect_gaps(asset, "1h")
    all_gaps.extend(gaps)

print(f"Total gaps detected across all assets: {len(all_gaps)}")
if all_gaps:
    gap_df = pd.DataFrame([g.model_dump() for g in all_gaps])
    display(gap_df.head(20))

In [ ]:
# Gap timeline visualisation (interactive Bokeh)
if all_gaps:
    gap_timeline = create_gap_timeline(all_gaps)
    show(gap_timeline)
else:
    print("No gaps detected — all data is continuous.")

In [ ]:
# Volume profile for the first asset (interactive Bokeh — pan/zoom enabled)
if assets:
    sample_asset = assets[0]
    ohlcv_df = loader.load_ohlcv(sample_asset, "1h")
    print(f"Volume profile for {sample_asset} (1h) — {len(ohlcv_df)} candles")
    vol_fig = create_volume_profile(ohlcv_df)
    show(vol_fig)

In [ ]:
# Asset filtering — keep assets with >= 730 days of data and <= 5% gap ratio
filter_results = coverage_analyzer.filter_assets(assets, "1h", min_days=730, max_gap_ratio=0.05)
filter_df = pd.DataFrame([r.model_dump() for r in filter_results])
display(filter_df)

selected_assets = [r.asset for r in filter_results if r.included]
print(f"\nSelected assets for further analysis ({len(selected_assets)}): {selected_assets}")

## 2. Bar Construction

Load aggregated bars (tick, volume, dollar, imbalance, run) for selected assets, compare bar counts per week, and analyse bar duration variability across bar types.

In [ ]:
# Load all available bar configurations for the first selected asset
analysis_asset = selected_assets[0] if selected_assets else assets[0]
bar_configs = loader.get_available_bar_configs(analysis_asset)
print(f"Bar configurations for {analysis_asset}: {len(bar_configs)}")
for bt, ch in bar_configs:
    print(f"  {bt} (config_hash={ch})")

# Load bar DataFrames keyed by bar_type
bar_data = {}
for bar_type, config_hash in bar_configs:
    df = loader.load_bars(analysis_asset, bar_type, config_hash)
    if not df.empty:
        bar_data[bar_type] = df
        print(f"  Loaded {bar_type}: {len(df)} bars")

print(f"\nBar types loaded: {list(bar_data.keys())}")

In [ ]:
# Weekly bar count histogram (interactive Bokeh)
if bar_data:
    weekly_counts = {}
    for bar_type, df in bar_data.items():
        counts = bar_comparator.bar_count_per_period(df, freq="W")
        weekly_counts[bar_type] = counts

    bar_hist = create_bar_count_histogram(weekly_counts)
    show(bar_hist)

    # Compare bar count variability across types
    variability_df = bar_comparator.compare_bar_count_variability(bar_data)
    print("Bar count variability (CV = std / mean):")
    display(variability_df)
else:
    print("No bar data available — run bar aggregation first (Phase 2).")

In [ ]:
# Bar duration statistics and boxplot (matplotlib)
if bar_data:
    duration_stats = []
    durations_dict = {}
    for bar_type, df in bar_data.items():
        stats = bar_comparator.compute_duration_stats(df, analysis_asset, bar_type)
        duration_stats.append(stats)
        durations_dict[bar_type] = (df["end_ts"] - df["start_ts"]).dt.total_seconds() / 60.0

    dur_stats_df = pd.DataFrame([s.model_dump() for s in duration_stats])
    print("Bar duration statistics (minutes):")
    display(dur_stats_df)

    fig = create_bar_duration_boxplot(durations_dict)
    plt.show()
    plt.close(fig)

## 3. Return Distributions

Compute log returns for each bar type, test normality (Jarque-Bera), examine QQ plots, and overlay return distributions. Fat tails and non-normality are expected for crypto — the question is whether alternative bars improve distributional properties.

In [ ]:
# Compute log returns per bar type
bar_returns = {}
for bar_type, df in bar_data.items():
    returns = return_analyzer.compute_log_returns(df)
    bar_returns[bar_type] = returns
    print(f"{bar_type}: {len(returns)} returns, mean={returns.mean():.6f}, std={returns.std():.6f}")

# Also compute returns from raw OHLCV (1h time bars) for comparison
ohlcv_1h = loader.load_ohlcv(analysis_asset, "1h")
if not ohlcv_1h.empty:
    time_returns = return_analyzer.compute_log_returns(ohlcv_1h)
    bar_returns["time_1h"] = time_returns
    print(f"time_1h: {len(time_returns)} returns, mean={time_returns.mean():.6f}, std={time_returns.std():.6f}")

In [ ]:
# Return statistics table (interactive Bokeh DataTable)
all_return_stats = []
for bar_type, returns in bar_returns.items():
    stat = return_analyzer.compute_statistics(returns, analysis_asset, bar_type)
    all_return_stats.append(stat)

stats_table = create_statistics_table(all_return_stats)
show(stats_table)

# Also display as pandas DataFrame for easy reading
stats_df = pd.DataFrame([s.model_dump() for s in all_return_stats])
display(stats_df)

In [ ]:
# QQ-plot grid — compare each bar type's returns against normal distribution
qq_data = {}
for bar_type, returns in bar_returns.items():
    theoretical, observed = return_analyzer.compute_qq_data(returns)
    qq_data[bar_type] = (theoretical, observed)

fig = create_qq_grid(qq_data)
plt.show()
plt.close(fig)

In [ ]:
# Return distribution overlay — histograms + KDE per bar type
fig = create_return_distribution(bar_returns)
plt.show()
plt.close(fig)

## 4. Autocorrelation & Serial Dependence

Examine ACF/PACF structure in raw returns (momentum or mean-reversion patterns) and squared returns (volatility clustering / ARCH effects). The Ljung-Box test formally tests for serial correlation. Per Lopez de Prado, information-driven bars should reduce spurious autocorrelation in returns while preserving volatility clustering.

In [ ]:
# ACF analysis on raw returns per bar type
acf_results = {}
for bar_type, returns in bar_returns.items():
    result = acf_analyzer.compute_acf_analysis(returns, analysis_asset, bar_type)
    acf_results[bar_type] = result
    print(
        f"{bar_type}: Ljung-Box Q={result.ljung_box_stat:.2f}, "
        f"p={result.ljung_box_pvalue:.4f}, "
        f"serial_corr={'YES' if result.has_serial_correlation else 'no'}"
    )

# ACF comparison grid (matplotlib)
acf_arrays = {bt: r.acf_values for bt, r in acf_results.items()}
fig = create_acf_comparison_grid(acf_arrays)
plt.show()
plt.close(fig)

In [ ]:
# ACF analysis on SQUARED returns (volatility clustering / ARCH effects)
squared_acf_results = {}
for bar_type, returns in bar_returns.items():
    result = acf_analyzer.compute_squared_acf(returns, analysis_asset, bar_type)
    squared_acf_results[bar_type] = result
    print(
        f"{bar_type} (r²): Ljung-Box Q={result.ljung_box_stat:.2f}, "
        f"p={result.ljung_box_pvalue:.4f}, "
        f"vol_clustering={'YES' if result.has_serial_correlation else 'no'}"
    )

# ACF comparison grid for squared returns
sq_acf_arrays = {bt: r.acf_values for bt, r in squared_acf_results.items()}
fig = create_acf_comparison_grid(sq_acf_arrays)
fig.suptitle("ACF of Squared Returns (Volatility Clustering)", fontsize=14)
plt.show()
plt.close(fig)

In [ ]:
# Ljung-Box summary table
ljung_box_rows = []
for bar_type in bar_returns:
    raw = acf_results[bar_type]
    sq = squared_acf_results[bar_type]
    ljung_box_rows.append(
        {
            "bar_type": bar_type,
            "LB_stat_returns": round(raw.ljung_box_stat, 2),
            "LB_pvalue_returns": round(raw.ljung_box_pvalue, 4),
            "serial_corr": raw.has_serial_correlation,
            "LB_stat_squared": round(sq.ljung_box_stat, 2),
            "LB_pvalue_squared": round(sq.ljung_box_pvalue, 4),
            "vol_clustering": sq.has_serial_correlation,
        }
    )

ljung_box_df = pd.DataFrame(ljung_box_rows)
print("Ljung-Box Test Summary (lag=40):")
display(ljung_box_df)

## 5. Conclusions & Go/No-Go Decisions

Summary comparison table and answers to the four research checkpoint questions.

In [ ]:
# Summary comparison table across bar types
summary_rows = []
for bar_type in bar_returns:
    stat = next(s for s in all_return_stats if s.bar_type == bar_type)
    acf_r = acf_results[bar_type]
    acf_sq = squared_acf_results[bar_type]
    summary_rows.append(
        {
            "bar_type": bar_type,
            "n_returns": stat.count,
            "mean": round(stat.mean, 6),
            "std": round(stat.std, 6),
            "skewness": round(stat.skewness, 3),
            "excess_kurtosis": round(stat.kurtosis, 3),
            "JB_pvalue": round(stat.jarque_bera_pvalue, 4),
            "is_normal": stat.is_normal,
            "serial_corr": acf_r.has_serial_correlation,
            "vol_clustering": acf_sq.has_serial_correlation,
        }
    )

summary_df = pd.DataFrame(summary_rows)
print("=== RC1 Summary Comparison Table ===\n")
display(summary_df)

### Q1: Is the data quality sufficient for modelling?

**Criteria:** Coverage >= 95% for selected assets, no gaps exceeding 48 hours, at least 2 years of hourly data.

**Decision:** _Fill in after running the notebook with real data._

---

### Q2: Do alternative bars (dollar, imbalance, run) improve distributional properties over time bars?

**Criteria:** Lower excess kurtosis, reduced skewness, closer to normality (higher JB p-value).

**Decision:** _Fill in after running the notebook with real data._

---

### Q3: Do information-driven bars reduce serial correlation in returns?

**Criteria:** Ljung-Box p-value > 0.05 for raw returns (no spurious autocorrelation), while squared returns still show clustering (ARCH effects present — a sign the bar preserves meaningful volatility structure).

**Decision:** _Fill in after running the notebook with real data._

---

### Q4: Which bar types should proceed to Phase 4 (feature engineering)?

**Recommendation:** Select bar types that:
1. Have sufficient sample size (>= 1000 bars)
2. Show reduced serial correlation vs time bars
3. Preserve volatility clustering (useful signal)
4. Have reasonable duration variability (CV not extreme)

**Selected bar types for Phase 4:** _Fill in after running the notebook with real data._

In [ ]:
# Clean up — dispose database connection
cm.dispose()
print("Database connection closed.")